# Clase 7 — Prompting, RAG y agentes

## Pregunta central

> **¿Cómo conectamos un modelo generativo con instrucciones, documentos y herramientas?**

## Idea principal

Prompting guía el comportamiento, RAG agrega evidencia al contexto y un agente selecciona acciones dentro de límites definidos por software.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Reconocer instrucciones zero-shot, few-shot y salidas estructuradas.
- Elegir inicialmente entre prompting, RAG y fine-tuning.
- Construir un mini-RAG local con embeddings y Qwen.
- Validar una salida JSON antes de usarla.
- Simular una llamada a herramienta con esquema y lista permitida.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Anatomía de un prompt |
| 2 | Prompting, RAG o fine-tuning |
| 3 | Pipeline RAG |
| 4 | Generación local y JSON |
| 5 | Agentes y function calling |
| 6 | Actividad de grounding |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** prompting avanzado, vector stores, reranking y agentes del Track Salud.

---
## 1. Anatomía de un prompt

Un LLM recibe una secuencia de tokens y genera tokens nuevos. El
prompt es todo el contenido que colocamos en esa secuencia para
orientar una inferencia.

```text
mensajes + contexto + ejemplos + pregunta
                  |
                  v
             tokenización
                  |
                  v
                 LLM
                  |
                  v
            tokens de salida
```

Modificar el prompt no cambia los pesos del modelo. Cambia la
información disponible en esa ejecución.

### Roles de los mensajes

Muchas interfaces organizan el prompt en mensajes:

| Rol | Función habitual |
|---|---|
| System | Reglas generales y límites del asistente |
| User | Solicitud y datos entregados por el usuario |
| Assistant | Respuestas anteriores o ejemplos |
| Tool | Resultado devuelto por una herramienta |

Los roles ayudan a estructurar el intercambio, pero no convierten
una instrucción en una garantía de seguridad.

### Prompt como contrato

Un prompt útil especifica:

```text
objetivo
    + datos de entrada
    + contexto permitido
    + restricciones
    + comportamiento si falta información
    + formato de salida
```

| Técnica | Qué significa | Cuándo ayuda | Ejemplo |
|---|---|---|---|
| Zero-shot | Instrucción sin ejemplos | La tarea es simple y conocida | “Clasificá el mensaje” |
| Few-shot | Incluye ejemplos de entrada/salida | El formato o criterio necesita demostración | Dos casos etiquetados |
| Descomposición | Separa una tarea en etapas | Cada etapa puede verificarse | Extraer y luego validar |
| Salida estructurada | Define un esquema | Otra función consumirá la salida | JSON con campos definidos |

Ejemplo few-shot:

```text
Entrada: "Quiero mover mi cita"
Salida:  {"categoria": "reprogramacion"}

Entrada: "No puedo ingresar al sistema"
Salida:  {"categoria": "soporte"}

Entrada nueva: "Necesito otra fecha"
Salida:
```

Los ejemplos no demuestran que el modelo aprendió permanentemente;
forman parte del contexto de esa inferencia.

Pedir “pensá paso a paso” no garantiza verdad. En software interesa
más poder verificar entradas, salidas, fuentes y acciones.

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Prompt | Mensajes e información entregados al modelo |
| Token | Unidad que procesa el modelo |
| Contexto | Tokens disponibles para producir la respuesta |
| Grounding | Vincular una respuesta con evidencia disponible |
| Chunk | Fragmento recuperable de un documento |
| Embedding | Vector usado para representar y comparar contenido |
| Vector store | Sistema que persiste y busca embeddings |
| Reranking | Segunda etapa que reordena candidatos |
| Tool/function calling | Solicitud estructurada para usar una función |
| Agente | Bucle controlado que selecciona acciones y observa resultados |
| Fine-tuning | Ajuste de pesos usando un dataset de entrenamiento |
| Schema | Definición de campos, tipos y restricciones |
| Alucinación | Contenido plausible que no está respaldado |
| Temperatura | Configuración que modifica variabilidad de generación |
| Similitud coseno | Medida de alineación entre dos vectores |
| TF-IDF | Representación lexical que pondera palabras frecuentes en un documento pero distintivas en la colección |
| Top-k | Los primeros `k` resultados de un ranking |

## Un ejemplo de contrato de salida

Una aplicación suele necesitar datos con campos conocidos, no un
párrafo libre. Por ejemplo:

```json
{
  "categoria": "reprogramacion",
  "prioridad": "normal",
  "requiere_revision": false
}
```

JSON válido todavía puede contener valores incorrectos. Hay dos
verificaciones principales y una tercera capa de negocio:

1. **sintáctica:** se puede parsear y respeta el esquema;
2. **semántica:** los valores están respaldados por la entrada;
3. **de negocio:** la aplicación permite esa categoría y esa acción.

```text
texto del modelo
      |
      v
parseo JSON -------- falla -> rechazar o reintentar
      |
      v
validación schema --- falla -> rechazar
      |
      v
reglas de negocio --- falla -> revisión
      |
      v
dato aceptado
```

---
## 2. ¿Prompting, RAG o fine-tuning?

Los tres enfoques intervienen en lugares diferentes:

```text
prompting   -> cambia el contenido de la entrada
RAG         -> busca evidencia y la agrega a la entrada
fine-tuning -> modifica pesos mediante entrenamiento
```

| Necesidad principal | Primer enfoque | Ejemplo | Qué no resuelve por sí solo |
|---|---|---|---|
| Cambiar instrucciones o formato | Prompting | Responder con JSON | Agregar conocimiento confiable |
| Responder con documentos actuales | RAG | Consultar un manual vigente | Cambiar profundamente el comportamiento |
| Especializar conducta repetida | Fine-tuning | Estilo de clasificación estable | Actualizar hechos dinámicos |

Pueden combinarse. Un RAG también necesita un prompt y un sistema
especializado puede usar un modelo ajustado.

### Qué decisión intenta resolver cada uno

**Prompting**

- Es rápido de cambiar.
- No requiere dataset de entrenamiento.
- Consume espacio de contexto.
- No garantiza obediencia ni incorpora hechos de forma confiable.

**RAG**

- Mantiene documentos fuera de los pesos.
- Permite actualizar fuentes sin reentrenar el LLM.
- Puede mostrar qué fragmentos recuperó.
- Agrega una nueva fuente de error: retrieval.

**Fine-tuning**

- Requiere ejemplos de entrenamiento y evaluación.
- Modifica los pesos para especializar comportamiento.
- No es una base de datos de hechos actuales.
- Tiene costo de entrenamiento, versionado y mantenimiento.

**RAG no reentrena el LLM.** Recupera fragmentos y los coloca en el
contexto de una inferencia.

Antes de elegir conviene preguntar:

1. ¿El problema es conocimiento faltante o comportamiento?
2. ¿La información cambia con frecuencia?
3. ¿Necesitamos mostrar fuentes?
4. ¿Tenemos ejemplos de calidad para entrenar?
5. ¿Cómo evaluaremos el resultado?

---
## 3. Mini-RAG local en cinco pasos

RAG significa **Retrieval-Augmented Generation**: generación
aumentada con recuperación. Separa el sistema en dos responsabilidades:

- retrieval decide qué evidencia entregar;
- generation redacta una respuesta usando esa evidencia.

### Etapa de preparación

```text
archivos
   |
   v
extracción de texto
   |
   v
división en chunks
   |
   v
embeddings
   |
   v
vector store + metadatos
```

Un **chunk** es un fragmento recuperable. Debe ser suficientemente
corto para ser específico y suficientemente largo para conservar
contexto.

| Chunk demasiado corto | Chunk demasiado largo |
|---|---|
| Puede perder definiciones y referencias | Puede mezclar varios temas |
| Recupera frases incompletas | Consume más tokens |
| Necesita contexto vecino | Reduce precisión del ranking |

Un vector store persiste embeddings y metadatos como documento,
sección, fecha o permisos. En esta clase usamos una matriz en memoria
para que el mecanismo sea visible.

### Etapa de consulta

```text
pregunta
   |
   v
embedding de pregunta
   |
   v
búsqueda por similitud
   |
   v
candidatos top-k
   |
   v
reranking opcional
   |
   v
prompt con evidencia -> LLM -> respuesta
```

Un **reranker** recibe pocos candidatos y vuelve a ordenarlos con un
modelo más preciso y más costoso. No lo ejecutamos en esta clase,
pero ubicamos su función.

Los documentos son ficticios. Esto permite preguntar por hechos que
el modelo no podría conocer de su preentrenamiento.

La búsqueda principal usa embeddings normalizados y calcula
**similitud coseno**: valores mayores indican vectores más alineados.
Si el encoder no está disponible, el notebook usa **TF-IDF** como
fallback local. TF-IDF representa qué palabras aparecen y cuánto
distinguen a un documento; encuentra coincidencias lexicales, pero
no ofrece la misma noción de similitud semántica que un encoder.

### Fuentes de error

| Etapa | Error posible |
|---|---|
| Ingesta | Texto mal extraído o documento desactualizado |
| Chunking | Respuesta dividida entre fragmentos |
| Embeddings | Pregunta y evidencia quedan lejos |
| Top-k | Entra ruido o falta el fragmento correcto |
| Generación | El LLM ignora o contradice el contexto |
| Aplicación | Presenta una respuesta sin fuente ni advertencia |

In [ ]:
import json
import os
import ssl
import time

import certifi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

FAST_MODE = True
RUN_LLM = True
SEED = 42
ssl._create_default_https_context = (
    lambda: ssl.create_default_context(cafile=certifi.where())
)

documentos = pd.DataFrame([
    [
        "garantia",
        (
            "Los sensores NubeAndina tienen una garantía de "
            "dieciocho meses desde la fecha de instalación."
        ),
    ],
    [
        "calibracion",
        (
            "La calibración preventiva se realiza cada seis meses. "
            "El procedimiento tarda aproximadamente 40 minutos."
        ),
    ],
    [
        "soporte",
        (
            "El soporte crítico atiende todos los días. Los casos "
            "no críticos se responden de lunes a viernes."
        ),
    ],
    [
        "exportacion",
        (
            "Los reportes pueden exportarse en CSV o PDF desde el "
            "panel de administración."
        ),
    ],
], columns=["id", "texto"])
documentos

In [ ]:
encoder = None
vectorizador_tfidf = None
try:
    from sentence_transformers import SentenceTransformer

    encoder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cpu",
    )
    modo_retrieval = "embeddings all-MiniLM-L6-v2"
except Exception as error:
    from sklearn.feature_extraction.text import TfidfVectorizer

    vectorizador_tfidf = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        norm="l2",
    )
    modo_retrieval = (
        "fallback TF-IDF local "
        f"({type(error).__name__})"
    )

def codificar_documentos(textos):
    if encoder is not None:
        return encoder.encode(
            list(textos),
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    return vectorizador_tfidf.fit_transform(textos).toarray()

def codificar_consultas(textos):
    if encoder is not None:
        return encoder.encode(
            list(textos),
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    return vectorizador_tfidf.transform(textos).toarray()

embeddings_documentos = codificar_documentos(
    documentos["texto"].tolist()
)
print("Modo de retrieval:", modo_retrieval)

def recuperar(pregunta, top_k=2, tabla=None, embeddings=None):
    tabla = documentos if tabla is None else tabla
    embeddings = (
        embeddings_documentos if embeddings is None else embeddings
    )
    vector = codificar_consultas([pregunta])[0]
    scores = embeddings @ vector
    indices = np.argsort(scores)[::-1][:top_k]
    resultado = tabla.iloc[indices].copy()
    resultado["score"] = scores[indices]
    return resultado.reset_index(drop=True)

pregunta = "¿Cuánto dura la garantía de los sensores?"
recuperados = recuperar(pregunta, top_k=2)
display(recuperados.round({"score": 3}))

plt.figure(figsize=(7, 3))
plt.barh(
    recuperados["id"][::-1],
    recuperados["score"][::-1],
    color=["#2a9d8f", "#8ecae6"],
)
plt.xlabel("Similitud coseno")
plt.title("Fragmentos recuperados")
plt.xlim(0, 1)
plt.show()

### Primera inspección

Antes de generar, leé el ranking. No conviene ocultar retrieval
detrás del LLM porque necesitamos diagnosticar cada etapa.

- ¿el primer fragmento contiene la respuesta?
- ¿`top_k=2` agregó ruido?
- ¿la pregunta comparte palabras o solo significado?
- ¿el score es alto en términos absolutos o solo el mayor disponible?

`top_k` controla un intercambio:

| `top_k` pequeño | `top_k` grande |
|---|---|
| Menos ruido y menos tokens | Más posibilidad de incluir evidencia |
| Puede omitir un fragmento necesario | Puede mezclar información irrelevante |

Si retrieval no entrega evidencia, la generación no puede repararlo
de manera confiable. Un LLM puede producir una respuesta plausible,
pero no recuperó mágicamente el documento faltante.

Después construimos un contexto con IDs de fuente. Mantener los IDs
fuera del texto generado permite que el pipeline conserve
trazabilidad aunque el modelo no cite correctamente.

In [ ]:
def construir_contexto(fragmentos):
    return "\n\n".join(
        f"[Fuente: {fila.id}]\n{fila.texto}"
        for fila in fragmentos.itertuples(index=False)
    )

contexto = construir_contexto(recuperados)
print(contexto)

---
## 4. Qwen local: comparar memoria paramétrica y evidencia

### Qué significa ejecutar un LLM local

“Local” significa que los pesos se cargan en la computadora y la
inferencia se ejecuta sin enviar el prompt a una API externa.

Usaremos Qwen 2.5 Instruct 0.5B cuantizado en Q4 mediante
`llama.cpp`. El archivo ocupa aproximadamente 500 MB y queda en
cache. El modelo es pequeño a propósito: sirve para observar el
pipeline, no para asumir calidad productiva.

| Término | Significado |
|---|---|
| 0.5B | Aproximadamente 500 millones de parámetros |
| Cuantización Q4 | Pesos representados con pocos bits para reducir memoria |
| GGUF | Formato de archivo usado por motores como llama.cpp |
| llama.cpp | Motor de inferencia optimizado para CPU |
| `n_ctx=2048` | Límite configurado de tokens de contexto |

La cuantización reduce memoria y puede acelerar CPU, a cambio de una
aproximación de los pesos. No cambia el hecho de que el modelo puede
cometer errores.

### Pesos frente a contexto

```text
pesos del modelo
    -> conocimiento estadístico del entrenamiento
    -> no cambia al hacer una pregunta

contexto del prompt
    -> instrucciones y evidencia de esta ejecución
    -> se descarta al terminar
```

Primero preguntaremos sin documentos. Después repetiremos la misma
pregunta agregando el contexto recuperado. La comparación muestra
qué aporta RAG.

Si no puede descargarse o cargarse, retrieval y el ejercicio de
agentes siguen siendo ejecutables y se muestra un fallback explícito.

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
llm = None

if RUN_LLM:
    try:
        from llama_cpp import Llama

        inicio = time.perf_counter()
        ruta_modelo = hf_hub_download(
            repo_id=REPO_ID,
            filename=FILENAME,
        )
        llm = Llama(
            model_path=ruta_modelo,
            n_ctx=2048,
            n_gpu_layers=0,
            n_threads=max(1, min(6, (os.cpu_count() or 2) // 2)),
            verbose=False,
            seed=SEED,
        )
        print(
            f"Modelo listo: {os.path.getsize(ruta_modelo) / 2**20:.1f} MB "
            f"en {time.perf_counter() - inicio:.1f} s"
        )
    except Exception as error:
        print("LLM no disponible; se usará un fallback didáctico.")
        print("Detalle:", type(error).__name__, str(error)[:180])
else:
    print("RUN_LLM=False: se usará un fallback didáctico.")

In [ ]:
def generar(pregunta_usuario, contexto_usuario=None, max_tokens=70):
    if llm is None:
        return {
            "texto": (
                "[Fallback: el LLM no está cargado. "
                "Revisá directamente los fragmentos recuperados.]"
                if contexto_usuario
                else "[Fallback: no se ejecutó generación local.]"
            ),
            "tokens_entrada": None,
            "segundos": 0.0,
            "modo": "fallback",
        }

    if contexto_usuario:
        mensaje = f'''Respondé usando únicamente el CONTEXTO.
Si falta la respuesta, respondé exactamente: "No está en los documentos".
Incluí el ID de la fuente entre corchetes.

CONTEXTO:
{contexto_usuario}

PREGUNTA:
{pregunta_usuario}'''
        sistema = "Sos un asistente breve que no inventa evidencia."
    else:
        mensaje = f"Respondé brevemente: {pregunta_usuario}"
        sistema = "Sos un asistente prudente."

    tokens_entrada = len(
        llm.tokenize(mensaje.encode("utf-8"), add_bos=True)
    )
    inicio = time.perf_counter()
    salida = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": sistema},
            {"role": "user", "content": mensaje},
        ],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return {
        "texto": salida["choices"][0]["message"]["content"].strip(),
        "tokens_entrada": tokens_entrada,
        "segundos": time.perf_counter() - inicio,
        "modo": "qwen-local",
    }

sin_contexto = generar(pregunta)
con_rag = generar(pregunta, contexto)

print("SIN CONTEXTO")
print(f"Modo: {sin_contexto['modo']}")
print(sin_contexto["texto"])
print("\nCON RAG")
print(f"Modo: {con_rag['modo']}")
print(con_rag["texto"])
print("\nFuentes recuperadas:", ", ".join(recuperados["id"]))
print("Tokens de entrada:", con_rag["tokens_entrada"])
print("Tiempo:", round(con_rag["segundos"], 2), "s")

### Qué debemos separar

| Elemento | Qué debemos verificar |
|---|---|
| Pregunta | Está clara y no contiene supuestos incorrectos |
| Retrieval | Recuperó el fragmento correcto |
| Contexto | Incluye evidencia suficiente y no contradictoria |
| Generación | No agrega hechos ausentes |
| Fuentes | El pipeline conserva IDs verificables |

- La respuesta sin contexto proviene de los pesos y puede inventar.
- La respuesta RAG debería poder contrastarse con los fragmentos.
- El pipeline conserva los IDs de fuente aunque el LLM omita citarlos.
- Un contexto más largo ocupa tokens y aumenta trabajo y KV cache.
- RAG reduce algunos errores; no garantiza verdad.

### Grounding

Grounding significa que las afirmaciones relevantes pueden vincularse
con evidencia permitida. No basta con que la respuesta “suene
correcta”.

```text
afirmación de la respuesta
          |
          v
¿aparece en una fuente recuperada?
     /                 \
   sí                   no
   |                    |
respaldada        rechazar o revisar
```

---
## 5. Salida JSON validada

Una persona puede interpretar una respuesta redactada de muchas
maneras. Una función necesita una estructura predecible. Por eso,
cuando la salida alimentará otro componente del sistema, conviene
acordar un contrato.

Supongamos que una consulta debe clasificarse. Un texto libre como
“parece un tema relacionado con la garantía” obliga a escribir
reglas frágiles para encontrar la categoría. En cambio, una salida
estructurada puede separar los datos:

```json
{
  "categoria": "garantia",
  "requiere_revision": false
}
```

### Qué describe un schema

Un **schema** especifica la forma permitida para esos datos:

| Elemento | Pregunta que responde | Ejemplo |
|---|---|---|
| Tipo raíz | ¿Qué estructura esperamos? | Un objeto JSON |
| Propiedades | ¿Qué campos pueden aparecer? | `categoria` |
| Tipos | ¿Qué clase de valor admite cada campo? | Texto o booleano |
| Campos requeridos | ¿Cuáles no pueden faltar? | Ambos campos |
| Valores permitidos | ¿Qué opciones acepta el negocio? | `garantia`, `soporte`, `otro` |

El flujo completo no termina cuando el modelo escribe llaves:

```text
consulta
   |
   v
LLM propone texto
   |
   v
parseo JSON
   |
   v
validación de schema
   |
   v
validación semántica y de negocio
   |
   v
aceptar, reintentar o enviar a revisión
```

En el experimento pedimos una clasificación y comprobamos tanto los
campos como sus tipos y valores permitidos. Si el modelo no produce
una salida aceptable, el código usa un fallback visible en lugar de
fingir que la inferencia fue correcta.

In [ ]:
esquema_clasificacion = {
    "type": "object",
    "properties": {
        "categoria": {
            "type": "string",
            "enum": ["garantia", "soporte", "otro"],
        },
        "requiere_revision": {"type": "boolean"},
    },
    "required": ["categoria", "requiere_revision"],
}

error_json = None
if llm is not None:
    try:
        salida_json = llm.create_chat_completion(
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Clasificá la consulta. Respondé solo JSON "
                        "según el esquema solicitado."
                    ),
                },
                {
                    "role": "user",
                    "content": "¿Qué cobertura tienen mis sensores?",
                },
            ],
            temperature=0.0,
            max_tokens=40,
            response_format={
                "type": "json_object",
                "schema": esquema_clasificacion,
            },
        )["choices"][0]["message"]["content"]
        clasificacion = json.loads(salida_json)
        assert set(clasificacion) == {
            "categoria", "requiere_revision"
        }
        assert clasificacion["categoria"] in {
            "garantia", "soporte", "otro"
        }
        assert isinstance(
            clasificacion["requiere_revision"], bool
        )
        modo_json = "Qwen local con schema"
    except Exception as error:
        error_json = f"{type(error).__name__}: {str(error)[:160]}"
        salida_json = json.dumps({
            "categoria": "garantia",
            "requiere_revision": False,
        })
        modo_json = "fallback validado: salida del LLM rechazada"
else:
    salida_json = json.dumps({
        "categoria": "garantia",
        "requiere_revision": False,
    })
    modo_json = "fallback validado: LLM no cargado"

clasificacion = json.loads(salida_json)
assert set(clasificacion) == {"categoria", "requiere_revision"}
assert clasificacion["categoria"] in {"garantia", "soporte", "otro"}
assert isinstance(clasificacion["requiere_revision"], bool)

print("Modo:", modo_json)
if error_json:
    print("Salida original rechazada:", error_json)
print("JSON parseable y validado:")
display(clasificacion)

### Cómo interpretar el resultado

`json.loads` responde solamente: “¿este texto tiene sintaxis JSON?”.
Los `assert` agregan controles simples sobre campos, tipos y valores.
En una aplicación real usaríamos una librería de schemas, mensajes de
error controlados y una política de reintentos.

Es importante distinguir tres resultados:

| Resultado | Ejemplo | Tratamiento |
|---|---|---|
| JSON inválido | Falta una comilla | Rechazar o reintentar |
| JSON válido, schema inválido | Falta un campo | Rechazar |
| JSON y schema válidos, contenido incorrecto | Categoría equivocada | Evaluar, corregir o revisar |

Un schema limita la **forma** de la respuesta. No demuestra que el
modelo entendió la consulta ni que el dato sea verdadero. Esa
diferencia evita delegar decisiones importantes a una comprobación
puramente sintáctica.

---
## 6. Agentes y function calling

Un LLM por sí solo produce texto. Para consultar una agenda, una base
de datos o un servicio externo necesita que software convencional
ejecute operaciones. **Function calling** es un protocolo: el modelo
propone el nombre de una función y argumentos estructurados; la
aplicación decide si acepta la propuesta.

El modelo no recibe acceso directo a la función ni a sus
credenciales:

```text
usuario
   |
   v
LLM propone: herramienta + argumentos
   |
   v
aplicación valida nombre, schema y permisos
   |
   +---- propuesta inválida ----> rechazar
   |
   v
herramienta ejecuta
   |
   v
resultado vuelve como observación
   |
   v
LLM redacta o propone el siguiente paso
```

Una **herramienta** es una función acotada que la aplicación expone,
por ejemplo `consultar_turnos(patient_id)`. Su definición debería
explicar qué hace, qué argumentos admite y qué devuelve.

### Propuesta no significa ejecución

El LLM puede inventar una herramienta, omitir argumentos o intentar
usar un identificador no permitido. Por eso la aplicación comprueba:

| Control | Pregunta |
|---|---|
| Lista permitida | ¿Existe y está habilitada esa herramienta? |
| Schema | ¿Los argumentos tienen nombres y tipos válidos? |
| Autorización | ¿Ese usuario puede operar sobre ese recurso? |
| Límites | ¿Se respetan tiempo, costo y cantidad de pasos? |
| Confirmación | ¿La acción tiene un efecto que una persona debe aprobar? |
| Registro | ¿Podemos reconstruir qué se propuso y ejecutó? |

Un **agente** agrega un bucle: observa un objetivo, propone una
acción, recibe el resultado y decide si necesita otro paso. Este
bucle no convierte al modelo en autónomo en sentido humano; sigue
siendo un programa con entradas, estados, herramientas y una
condición de finalización.

La aplicación, no el modelo, debe controlar credenciales, permisos,
timeouts, cantidad de pasos y efectos laterales. En esta práctica no
conectamos SQL ni servicios: simulamos una herramienta local de solo
lectura y validamos cada campo antes de ejecutarla.

In [ ]:
herramientas_permitidas = {"consultar_turnos"}
agenda_ficticia = {
    "P-102": [
        {"fecha": "2026-08-04", "especialidad": "clínica"},
    ],
}

solicitud_herramienta = {
    "tool": "consultar_turnos",
    "arguments": {"patient_id": "P-102"},
}

def ejecutar_solicitud_segura(solicitud):
    assert set(solicitud) == {"tool", "arguments"}
    assert solicitud["tool"] in herramientas_permitidas
    assert set(solicitud["arguments"]) == {"patient_id"}
    patient_id = solicitud["arguments"]["patient_id"]
    assert isinstance(patient_id, str) and len(patient_id) <= 12
    return {
        "patient_id": patient_id,
        "turnos": agenda_ficticia.get(patient_id, []),
        "origen": "agenda_ficticia_local",
    }

observacion = ejecutar_solicitud_segura(solicitud_herramienta)
display(pd.Series({
    "solicitud propuesta": json.dumps(
        solicitud_herramienta, ensure_ascii=False
    ),
    "observación": json.dumps(observacion, ensure_ascii=False),
    "efecto externo": "ninguno",
}).to_frame("simulación"))

### Cómo leer la simulación

`solicitud_herramienta` representa una propuesta que podría haber
producido el modelo. `ejecutar_solicitud_segura` representa la
frontera de confianza de la aplicación:

1. exige exactamente los campos esperados;
2. comprueba que la herramienta esté permitida;
3. valida nombre y tipo del argumento;
4. ejecuta sobre datos ficticios;
5. devuelve el origen junto con la observación.

No hay efecto externo. Esta separación permite estudiar el protocolo
sin conectar credenciales ni datos reales.

### ¿Qué falta para un agente conectado a sistemas reales?

- que el modelo elija entre herramientas mediante schemas;
- autenticación y autorización por usuario;
- límites de pasos, costo y tiempo;
- confirmación antes de acciones importantes;
- trazas y evaluación del recorrido completo;
- protección frente a instrucciones maliciosas en documentos.

También hacen falta fallbacks: qué ocurre si la herramienta no
responde, devuelve datos incompletos o el modelo entra en un ciclo.
Esos controles importan tanto como la calidad del modelo.

```text
agente útil = modelo
             + herramientas bien definidas
             + permisos mínimos
             + validaciones
             + límites
             + observabilidad
             + fallback humano
```

---
## Actividad — agregar conocimiento y verificar grounding

Modificá el documento, la pregunta y `TOP_K`. El objetivo no es
obtener una respuesta elegante: es poder señalar la evidencia exacta
y determinar en qué etapa aparece un error.

### Procedimiento sugerido

1. Ejecutá el ejemplo sin modificar y verificá qué fragmento queda
   primero.
2. Cambiá `NUEVO_TEXTO` manteniendo un hecho concreto y verificable.
3. Escribí una pregunta cuya respuesta aparezca de forma explícita.
4. Probá `TOP_K = 1`, `2` y `4`.
5. Compará evidencia, respuesta, cantidad de ruido y tokens.
6. Hacé una pregunta que ningún documento pueda responder.

Antes de aceptar una respuesta, completá mentalmente este flujo:

```text
¿se recuperó la fuente correcta?
       |
 no ---+--- sí
 |          |
error     ¿la respuesta usa solo esa evidencia?
retrieval       |
           no ---+--- sí
           |          |
        error      respuesta respaldada
      generación
```

In [ ]:
# TODO: cambiá estos tres valores.
NUEVO_TEXTO = (
    "La capacitación de seguridad se realiza el primer martes "
    "de cada mes y dura 90 minutos."
)
PREGUNTA_ACTIVIDAD = "¿Cuánto dura la capacitación de seguridad?"
TOP_K = 2

documentos_actividad = pd.concat([
    documentos,
    pd.DataFrame([{
        "id": "capacitacion",
        "texto": NUEVO_TEXTO,
    }]),
], ignore_index=True)
embeddings_actividad = codificar_documentos(
    documentos_actividad["texto"].tolist()
)
fuentes_actividad = recuperar(
    PREGUNTA_ACTIVIDAD,
    top_k=TOP_K,
    tabla=documentos_actividad,
    embeddings=embeddings_actividad,
)
display(fuentes_actividad.round({"score": 3}))

evidencia = construir_contexto(fuentes_actividad)
respuesta_actividad = generar(PREGUNTA_ACTIVIDAD, evidencia)
print("\nRespuesta:", respuesta_actividad["texto"])
print("Fuentes:", ", ".join(fuentes_actividad["id"]))

### Lista de verificación

- [ ] El fragmento correcto aparece en `top_k`.
- [ ] La respuesta no agrega hechos ausentes.
- [ ] Podemos identificar la fuente sin confiar en que el LLM la cite.
- [ ] Si cambiamos la pregunta por algo ausente, el sistema rechaza.
- [ ] Distinguimos un error de retrieval de un error de generación.

Registrá además:

| Prueba | `top_k` | ¿fuente correcta? | ¿respuesta respaldada? |
|---|---:|---|---|
| Pregunta presente | 1 | | |
| Pregunta presente | 4 | | |
| Pregunta ausente | 2 | | |

Una evaluación de RAG no debería reducirse a “me gustó la
respuesta”. Debe medir retrieval, fidelidad a las fuentes y utilidad
de la respuesta como dimensiones separadas.

---

## Síntesis de la clase

- Prompting guía una inferencia; no agrega conocimiento confiable.
- RAG recupera evidencia y la incorpora al contexto sin reentrenar.
- Retrieval, generación y grounding se evalúan por separado.
- JSON debe validarse antes de ser consumido por software.
- Un agente necesita herramientas acotadas, permisos y observabilidad.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 8 integra imágenes multibanda, coordenadas, polígonos e índices en un pipeline geoespacial local.

## Conexión con los tracks

Track Salud profundizará prompting, RAG, reranking, agentes con function calling/SQL y sus evaluaciones; aquí construimos el mapa común.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.